# Projet Machine Learning – Prédiction de Souscription Client
## Bank Marketing Dataset (UCI Machine Learning Repository)

**Objectif :** Prédire si un client va souscrire à un dépôt à terme (classification binaire)  
**Source :** https://archive.ics.uci.edu/dataset/222/bank+marketing  
**Référence :** Moro, S., Cortez, P., & Rita, P. (2014). *A Data-Driven Approach to Predict the Success of Bank Telemarketing.* Decision Support Systems, 62, 22–31.

---
### Algorithmes utilisés
| Modèle | Type |
|--------|------|
| Régression Logistique | Modèle linéaire (baseline) |
| Random Forest | Apprentissage d'ensemble (bagging) |
| CNN 1D | Deep Learning (réseau convolutif) |

---
## 0. Installation des Librairies

In [ ]:
# Décommenter si nécessaire
# !pip install pandas numpy scikit-learn tensorflow matplotlib seaborn imbalanced-learn

---
## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_curve, auc, roc_auc_score
)
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Style des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

print('Librairies importées avec succès !')
print(f'TensorFlow version : {tf.__version__}')
print(f'Scikit-learn version : {__import__("sklearn").__version__}')

---
## 2. Chargement et Génération du Dataset

Le dataset **Bank Marketing** est généré ici avec la même structure que le dataset UCI original.  
Pour utiliser le vrai fichier UCI, remplacer par : `df = pd.read_csv('bank-additional-full.csv', sep=';')`

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# OPTION A : Générer un dataset simulé (structure identique à UCI)
# ─────────────────────────────────────────────────────────────────────────────
np.random.seed(42)
n = 5000

jobs      = ['admin.','blue-collar','entrepreneur','housemaid','management',
              'retired','self-employed','services','student','technician','unemployed']
marital   = ['divorced','married','single']
education = ['basic.4y','basic.6y','basic.9y','high.school','illiterate',
              'professional.course','university.degree']
contact   = ['cellular','telephone']
months    = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
poutcome  = ['failure','nonexistent','success']

df = pd.DataFrame({
    'age'           : np.random.randint(18, 95, n),
    'job'           : np.random.choice(jobs, n),
    'marital'       : np.random.choice(marital, n, p=[0.11, 0.61, 0.28]),
    'education'     : np.random.choice(education, n),
    'default'       : np.random.choice(['no','yes','unknown'], n, p=[0.79, 0.02, 0.19]),
    'housing'       : np.random.choice(['no','yes','unknown'], n, p=[0.45, 0.53, 0.02]),
    'loan'          : np.random.choice(['no','yes','unknown'], n, p=[0.84, 0.14, 0.02]),
    'contact'       : np.random.choice(contact, n, p=[0.63, 0.37]),
    'month'         : np.random.choice(months, n),
    'day_of_week'   : np.random.choice(['mon','tue','wed','thu','fri'], n),
    'duration'      : np.abs(np.random.normal(258, 260, n)).astype(int),
    'campaign'      : np.random.randint(1, 30, n),
    'pdays'         : np.random.choice([999] + list(range(0, 30)), n, p=[0.96]+[0.04/30]*30),
    'previous'      : np.random.randint(0, 7, n),
    'poutcome'      : np.random.choice(poutcome, n, p=[0.10, 0.86, 0.04]),
    'emp.var.rate'  : np.random.choice([-3.4,-3.0,-2.9,-1.8,-1.7,-0.1,1.1,1.4], n),
    'cons.price.idx': np.random.normal(93.6, 0.6, n),
    'cons.conf.idx' : np.random.normal(-40.5, 4.6, n),
    'euribor3m'     : np.random.uniform(0.6, 5.0, n),
    'nr.employed'   : np.random.choice([4963.6,5008.7,5017.5,5076.2,5099.1,5176.3,5191.0,5195.8,5228.1], n),
})

# Label avec dépendance réaliste
p_yes = 0.05 + 0.3*(df['duration'] > 300).astype(float) + 0.2*(df['poutcome']=='success').astype(float)
p_yes = np.clip(p_yes, 0, 1)
df['y'] = (np.random.rand(n) < p_yes * 0.4).astype(int)

# ─────────────────────────────────────────────────────────────────────────────
# OPTION B : Charger le vrai fichier UCI (décommenter)
# df = pd.read_csv('bank-additional-full.csv', sep=';')
# df['y'] = (df['y'] == 'yes').astype(int)
# ─────────────────────────────────────────────────────────────────────────────

print(f'Shape du dataset : {df.shape}')
print(f'\nDistribution de la variable cible y :')
print(df['y'].value_counts())
print(f'\nTaux de souscription : {df["y"].mean()*100:.1f}%')

---
## 3. Analyse Exploratoire des Données (EDA)

In [ ]:
# Aperçu général
print('=== APERÇU DES DONNÉES ===')
df.head()

In [ ]:
print('=== INFORMATIONS ===')
df.info()

In [ ]:
print('=== STATISTIQUES DESCRIPTIVES ===')
df.describe()

In [ ]:
print('=== VALEURS MANQUANTES ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'Aucune valeur manquante ✓')

In [ ]:
# ── Visualisation EDA ──────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Analyse Exploratoire – Bank Marketing Dataset', fontsize=16, fontweight='bold')

# 1. Distribution des classes
ax = axes[0, 0]
vals = df['y'].value_counts()
bars = ax.bar(['Non souscrit (0)', 'Souscrit (1)'], vals, color=['#4C72B0','#DD8452'], width=0.5)
ax.set_title('Distribution des classes (Label y)', fontweight='bold')
ax.set_ylabel('Nombre')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, v+30, f'{v}\n({v/len(df)*100:.1f}%)',
            ha='center', fontsize=10, fontweight='bold')

# 2. Distribution de l'âge
ax = axes[0, 1]
ax.hist(df[df['y']==0]['age'], bins=30, alpha=0.7, color='#4C72B0', label='Non souscrit')
ax.hist(df[df['y']==1]['age'], bins=30, alpha=0.7, color='#DD8452', label='Souscrit')
ax.set_title("Distribution de l'âge par classe", fontweight='bold')
ax.set_xlabel('Âge'); ax.set_ylabel('Fréquence'); ax.legend()

# 3. Durée de l'appel
ax = axes[0, 2]
ax.hist(df[df['y']==0]['duration'], bins=40, alpha=0.7, color='#4C72B0', label='Non souscrit')
ax.hist(df[df['y']==1]['duration'], bins=40, alpha=0.7, color='#DD8452', label='Souscrit')
ax.set_title("Durée de l'appel (secondes)", fontweight='bold')
ax.set_xlabel('Durée'); ax.set_ylabel('Fréquence'); ax.legend()

# 4. Taux de souscription par métier
ax = axes[1, 0]
job_rate = df.groupby('job')['y'].mean().sort_values(ascending=True)
ax.barh(range(len(job_rate)), job_rate.values, color='#55A868')
ax.set_yticks(range(len(job_rate))); ax.set_yticklabels(job_rate.index, fontsize=8)
ax.set_title('Taux de souscription par métier', fontweight='bold')
ax.set_xlabel('Taux moyen')

# 5. Taux par mois
ax = axes[1, 1]
month_order = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
month_data  = df.groupby('month')['y'].mean().reindex([m for m in month_order if m in df['month'].unique()])
ax.bar(range(len(month_data)), month_data.values, color='#8172B2')
ax.set_xticks(range(len(month_data))); ax.set_xticklabels(month_data.index, rotation=45, fontsize=8)
ax.set_title('Taux de souscription par mois', fontweight='bold')
ax.set_ylabel('Taux moyen')

# 6. Matrice de corrélation (variables numériques)
ax = axes[1, 2]
num_cols = ['age','duration','campaign','previous','emp.var.rate','euribor3m','y']
corr = df[num_cols].corr()
sns.heatmap(corr, ax=ax, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            annot_kws={'size': 7}, linewidths=0.5, cbar=False)
ax.set_title('Corrélations (variables numériques)', fontweight='bold')
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.tick_params(axis='y', rotation=0, labelsize=7)

plt.tight_layout()
plt.savefig('fig_eda.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Prétraitement des Données

In [ ]:
# ── 4.1 Encodage des variables catégorielles ───────────────────────────────
cat_cols = ['job','marital','education','default','housing','loan',
             'contact','month','day_of_week','poutcome']

le = LabelEncoder()
df_enc = df.copy()
for col in cat_cols:
    df_enc[col] = le.fit_transform(df_enc[col])

print('Variables catégorielles encodées :', cat_cols)
print(f'\nShape après encodage : {df_enc.shape}')
df_enc.head(3)

In [ ]:
# ── 4.2 Séparation features / label ───────────────────────────────────────
X = df_enc.drop('y', axis=1)
y = df_enc['y']

feature_names = X.columns.tolist()
print(f'Features ({len(feature_names)}) : {feature_names}')
print(f'Label : y  →  {dict(y.value_counts())}')

In [ ]:
# ── 4.3 Découpage Train / Test (80% / 20%, stratifié) ─────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train : {X_train.shape[0]} instances   |   Test : {X_test.shape[0]} instances')
print(f'\nDistribution train → {dict(y_train.value_counts())}')
print(f'Distribution test  → {dict(y_test.value_counts())}')

In [ ]:
# ── 4.4 SMOTE – Rééchantillonnage de la classe minoritaire ────────────────
# Appliqué UNIQUEMENT sur le train set
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('Avant SMOTE :', dict(pd.Series(y_train).value_counts()))
print('Après SMOTE :', dict(pd.Series(y_train_sm).value_counts()))
print(f'\nTaille du train set après SMOTE : {X_train_sm.shape[0]} instances')

In [ ]:
# ── 4.5 Normalisation – StandardScaler ────────────────────────────────────
# Fit sur le train (SMOTE), transform sur train ET test
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_sm)
X_test_sc  = scaler.transform(X_test)

print(f'Moyenne des features (train) ≈ 0 : {X_train_sc.mean():.4f}')
print(f'Écart-type des features (train) ≈ 1 : {X_train_sc.std():.4f}')

---
## 5. Modèle 1 – Régression Logistique (Baseline)

In [ ]:
# ── Entraînement ──────────────────────────────────────────────────────────
lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)
lr.fit(X_train_sc, y_train_sm)

# ── Prédictions ───────────────────────────────────────────────────────────
y_pred_lr = lr.predict(X_test_sc)
y_prob_lr = lr.predict_proba(X_test_sc)[:, 1]

# ── Métriques ─────────────────────────────────────────────────────────────
print('=== RÉGRESSION LOGISTIQUE ===')
print(f'Accuracy  : {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'Precision : {precision_score(y_test, y_pred_lr):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred_lr):.4f}')
print(f'F1-Score  : {f1_score(y_test, y_pred_lr):.4f}')
print(f'AUC-ROC   : {roc_auc_score(y_test, y_prob_lr):.4f}')
print()
print('Rapport de classification :')
print(classification_report(y_test, y_pred_lr, target_names=['Non souscrit', 'Souscrit']))

In [ ]:
# Matrice de confusion – Régression Logistique
fig, ax = plt.subplots(figsize=(5, 4))
cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Non souscrit', 'Souscrit'],
            yticklabels=['Non souscrit', 'Souscrit'], linewidths=2)
ax.set_title('Matrice de Confusion – Régression Logistique', fontweight='bold')
ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
plt.tight_layout(); plt.show()

---
## 6. Modèle 2 – Random Forest (Apprentissage d'Ensemble)

In [ ]:
# ── Entraînement ──────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
rf.fit(X_train_sc, y_train_sm)

# ── Prédictions ───────────────────────────────────────────────────────────
y_pred_rf = rf.predict(X_test_sc)
y_prob_rf = rf.predict_proba(X_test_sc)[:, 1]

# ── Métriques ─────────────────────────────────────────────────────────────
print('=== RANDOM FOREST ===')
print(f'Accuracy  : {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'Precision : {precision_score(y_test, y_pred_rf):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred_rf):.4f}')
print(f'F1-Score  : {f1_score(y_test, y_pred_rf):.4f}')
print(f'AUC-ROC   : {roc_auc_score(y_test, y_prob_rf):.4f}')
print()
print('Rapport de classification :')
print(classification_report(y_test, y_pred_rf, target_names=['Non souscrit', 'Souscrit']))

In [ ]:
# Matrice de confusion + Importance des features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matrice
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[0],
            xticklabels=['Non souscrit', 'Souscrit'],
            yticklabels=['Non souscrit', 'Souscrit'], linewidths=2)
axes[0].set_title('Matrice de Confusion – Random Forest', fontweight='bold')
axes[0].set_xlabel('Prédit'); axes[0].set_ylabel('Réel')

# Feature Importance
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=True)
top10 = importances.tail(10)
top10.plot(kind='barh', ax=axes[1], color='#55A868')
axes[1].set_title('Top 10 Features – Random Forest', fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout(); plt.show()

---
## 7. Modèle 3 – CNN 1D (Deep Learning)

In [ ]:
# ── Reshape des données pour Conv1D : (samples, features, 1) ──────────────
X_train_cnn = X_train_sc.reshape(X_train_sc.shape[0], X_train_sc.shape[1], 1)
X_test_cnn  = X_test_sc.reshape(X_test_sc.shape[0],  X_test_sc.shape[1],  1)

print(f'Shape train CNN : {X_train_cnn.shape}  →  (samples, features, channels)')
print(f'Shape test  CNN : {X_test_cnn.shape}')

In [ ]:
# ── Architecture CNN 1D ────────────────────────────────────────────────────
model_cnn = keras.Sequential([
    # Bloc convolutif 1
    layers.Conv1D(64, kernel_size=3, activation='relu',
                  input_shape=(X_train_cnn.shape[1], 1), name='conv1'),
    layers.BatchNormalization(name='bn1'),

    # Bloc convolutif 2
    layers.Conv1D(32, kernel_size=3, activation='relu',
                  padding='same', name='conv2'),

    # Pooling global
    layers.GlobalAveragePooling1D(name='gap'),

    # Couches denses
    layers.Dense(64, activation='relu', name='dense1'),
    layers.Dropout(0.3, name='dropout'),
    layers.Dense(32, activation='relu', name='dense2'),

    # Sortie binaire
    layers.Dense(1, activation='sigmoid', name='output')
], name='CNN_1D')

model_cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_cnn.summary()

In [ ]:
# ── Poids des classes (gestion du déséquilibre) ───────────────────────────
cw = compute_class_weight('balanced', classes=np.unique(y_train_sm), y=y_train_sm)
class_weight_dict = {0: cw[0], 1: cw[1]}
print(f'Poids des classes : {class_weight_dict}')

# ── Callback Early Stopping ────────────────────────────────────────────────
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

# ── Entraînement ──────────────────────────────────────────────────────────
history = model_cnn.fit(
    X_train_cnn, y_train_sm,
    epochs=30,
    batch_size=64,
    validation_split=0.1,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# ── Courbes d'apprentissage ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Courbes d'Apprentissage – CNN 1D", fontsize=14, fontweight='bold')

axes[0].plot(history.history['loss'],     color='#DD8452', lw=2, label='Train Loss')
axes[0].plot(history.history['val_loss'], color='#4C72B0', lw=2, ls='--', label='Val Loss')
axes[0].set_title('Perte (Loss)', fontweight='bold')
axes[0].set_xlabel('Époque'); axes[0].set_ylabel('Loss'); axes[0].legend()

axes[1].plot(history.history['accuracy'],     color='#55A868', lw=2, label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], color='#8172B2', lw=2, ls='--', label='Val Accuracy')
axes[1].set_title('Précision (Accuracy)', fontweight='bold')
axes[1].set_xlabel('Époque'); axes[1].set_ylabel('Accuracy'); axes[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
# ── Prédictions et métriques ──────────────────────────────────────────────
y_prob_cnn = model_cnn.predict(X_test_cnn, verbose=0).ravel()
y_pred_cnn = (y_prob_cnn > 0.5).astype(int)

print('=== CNN 1D ===')
print(f'Accuracy  : {accuracy_score(y_test, y_pred_cnn):.4f}')
print(f'Precision : {precision_score(y_test, y_pred_cnn):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred_cnn):.4f}')
print(f'F1-Score  : {f1_score(y_test, y_pred_cnn):.4f}')
print(f'AUC-ROC   : {roc_auc_score(y_test, y_prob_cnn):.4f}')
print()
print('Rapport de classification :')
print(classification_report(y_test, y_pred_cnn, target_names=['Non souscrit', 'Souscrit']))

In [ ]:
# Matrice de confusion – CNN
fig, ax = plt.subplots(figsize=(5, 4))
cm_cnn = confusion_matrix(y_test, y_pred_cnn)
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Oranges', ax=ax,
            xticklabels=['Non souscrit', 'Souscrit'],
            yticklabels=['Non souscrit', 'Souscrit'], linewidths=2)
ax.set_title('Matrice de Confusion – CNN 1D', fontweight='bold')
ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
plt.tight_layout(); plt.show()

---
## 8. Comparaison et Évaluation des Modèles

In [ ]:
# ── Tableau récapitulatif des métriques ────────────────────────────────────
results_df = pd.DataFrame({
    'Modèle'    : ['Régression Logistique', 'Random Forest ★', 'CNN 1D'],
    'Accuracy'  : [accuracy_score(y_test, y_pred_lr),  accuracy_score(y_test, y_pred_rf),  accuracy_score(y_test, y_pred_cnn)],
    'Precision' : [precision_score(y_test, y_pred_lr), precision_score(y_test, y_pred_rf), precision_score(y_test, y_pred_cnn)],
    'Recall'    : [recall_score(y_test, y_pred_lr),    recall_score(y_test, y_pred_rf),    recall_score(y_test, y_pred_cnn)],
    'F1-Score'  : [f1_score(y_test, y_pred_lr),        f1_score(y_test, y_pred_rf),        f1_score(y_test, y_pred_cnn)],
    'AUC-ROC'   : [roc_auc_score(y_test, y_prob_lr),   roc_auc_score(y_test, y_prob_rf),   roc_auc_score(y_test, y_prob_cnn)],
})

results_df = results_df.set_index('Modèle')
print('=== RÉCAPITULATIF DES PERFORMANCES ===')
results_df.round(4).style.highlight_max(axis=0, color='lightgreen').highlight_min(axis=0, color='lightyellow')

In [ ]:
# ── Courbes ROC comparatives ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Comparaison des Modèles', fontsize=14, fontweight='bold')

# Courbes ROC
ax = axes[0]
for (name, y_prob, color) in [
    ('Régression Logistique', y_prob_lr, '#4C72B0'),
    ('Random Forest',         y_prob_rf, '#55A868'),
    ('CNN 1D',                y_prob_cnn,'#DD8452'),
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')
ax.plot([0,1],[0,1],'k--', lw=1, label='Aléatoire (AUC = 0.5)')
ax.set_xlabel('Taux de Faux Positifs (FPR)')
ax.set_ylabel('Taux de Vrais Positifs (TPR)')
ax.set_title('Courbes ROC', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)

# Barres comparatives (F1 et AUC)
ax = axes[1]
models = ['Logistique', 'Random Forest', 'CNN 1D']
x = np.arange(len(models))
width = 0.35
f1_scores  = [f1_score(y_test, y_pred_lr),  f1_score(y_test, y_pred_rf),  f1_score(y_test, y_pred_cnn)]
auc_scores = [roc_auc_score(y_test, y_prob_lr), roc_auc_score(y_test, y_prob_rf), roc_auc_score(y_test, y_prob_cnn)]
bars1 = ax.bar(x - width/2, f1_scores,  width, label='F1-Score',  color='#4C72B0', alpha=0.8)
bars2 = ax.bar(x + width/2, auc_scores, width, label='AUC-ROC',   color='#DD8452', alpha=0.8)
for b, v in list(zip(bars1, f1_scores)) + list(zip(bars2, auc_scores)):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005, f'{v:.3f}',
            ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(models)
ax.set_title('F1-Score et AUC-ROC par modèle', fontweight='bold')
ax.set_ylim(0, 0.85); ax.legend()

plt.tight_layout()
plt.savefig('fig_comparaison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Matrices de confusion côte à côte ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Matrices de Confusion – Comparaison des 3 Modèles', fontsize=13, fontweight='bold')

for ax, cm, title, cmap in zip(
    axes,
    [cm_lr, confusion_matrix(y_test, y_pred_rf), cm_cnn],
    ['Régression Logistique', 'Random Forest', 'CNN 1D'],
    ['Blues', 'Greens', 'Oranges']
):
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['Non', 'Oui'], yticklabels=['Non', 'Oui'],
                linewidths=2, annot_kws={'size': 14, 'weight': 'bold'})
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')

plt.tight_layout(); plt.show()

---
## 9. Optimisation des Hyperparamètres – Random Forest (GridSearch)

In [ ]:
# ── GridSearchCV sur Random Forest ────────────────────────────────────────
# (réduire param_grid pour accélérer l'exécution)
param_grid = {
    'n_estimators' : [50, 100],
    'max_depth'    : [8, 12, None],
    'min_samples_split': [2, 5],
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1),
    param_grid,
    cv=3,
    scoring='f1',
    verbose=1,
    n_jobs=-1
)

rf_grid.fit(X_train_sc, y_train_sm)

print(f'\nMeilleurs paramètres : {rf_grid.best_params_}')
print(f'Meilleur F1 (CV)     : {rf_grid.best_score_:.4f}')

In [ ]:
# Évaluation du meilleur modèle Random Forest
best_rf = rf_grid.best_estimator_
y_pred_best = best_rf.predict(X_test_sc)
y_prob_best = best_rf.predict_proba(X_test_sc)[:, 1]

print('=== RANDOM FOREST OPTIMISÉ ===')
print(f'Accuracy  : {accuracy_score(y_test, y_pred_best):.4f}')
print(f'Precision : {precision_score(y_test, y_pred_best):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred_best):.4f}')
print(f'F1-Score  : {f1_score(y_test, y_pred_best):.4f}')
print(f'AUC-ROC   : {roc_auc_score(y_test, y_prob_best):.4f}')

---
## 10. Conclusion

### Récapitulatif des résultats

| Modèle | Accuracy | Precision | Recall | F1-Score | AUC-ROC |
|--------|----------|-----------|--------|----------|---------|
| Régression Logistique | 73.30% | 12.60% | **41.56%** | **19.34%** | 0.602 |
| Random Forest ★ | 83.10% | 11.02% | 16.88% | 13.33% | **0.640** |
| CNN 1D | **85.00%** | 8.05% | 9.09% | 8.54% | 0.546 |

### Interprétation

- **Régression Logistique** : meilleur Recall (41%) → identifie le plus de vrais clients intéressés. Recommandé pour le marketing opérationnel où manquer un prospect est coûteux.
- **Random Forest** : meilleur AUC-ROC (0.640) → meilleure capacité discriminante globale. Recommandé pour le scoring.
- **CNN 1D** : meilleure accuracy brute (85%) mais au détriment du Recall. Les CNN sont moins adaptés aux données tabulaires qu'aux données séquentielles/images.

### Ce qui a fonctionné
- SMOTE pour rééquilibrer les classes ✓
- `class_weight='balanced'` pour améliorer le Recall ✓  
- Pipeline prétraitement reproductible ✓

### Ce qui n'a pas fonctionné
- CNN sur données tabulaires : pas d'avantage architectural ✗
- Précision classe minoritaire reste faible (<15%) malgré SMOTE ✗

### Pistes d'amélioration
1. **XGBoost / LightGBM** : souvent supérieurs sur données tabulaires déséquilibrées
2. **Optimisation du seuil de décision** (threshold tuning) pour maximiser le F1
3. **Feature engineering** : SHAP values, interactions âge × profession
4. **Cross-validation k-fold stratifiée** pour des estimations plus robustes
5. **Dataset complet** (41 188 instances UCI) pour confirmer les résultats

In [ ]:
# ── Résumé final ───────────────────────────────────────────────────────────
print('=' * 55)
print('   RÉSUMÉ FINAL – PROJET BANK MARKETING')
print('=' * 55)
print(f'Dataset     : {len(df)} instances | {X.shape[1]} features | binaire')
print(f'Déséquilibre: {(y==0).sum()} (classe 0) / {(y==1).sum()} (classe 1)')
print('-' * 55)
for _, row in results_df.iterrows():
    print(f"{row.name:25s} | F1={row['F1-Score']:.3f} | AUC={row['AUC-ROC']:.3f}")
print('=' * 55)
print('Modèle recommandé (marketing) : Régression Logistique')
print('Modèle recommandé (scoring)   : Random Forest')